# 🫁 ResNet-50 CXR — Clasificación Multilabel de Patologías Cardiopulmonares
## TFM: Sistema de Apoyo a la Decisión Clínica Multimodal · Módulo de Imagen
### Universidad de Salamanca · Máster en Análisis Avanzado de Datos Multivariantes y Big Data

---

## 📋 Descripción general

Este notebook implementa el **modelo de imagen** (radiografías de tórax, CXR) del sistema multimodal para predicción de patologías cardiopulmonares sobre el dataset **Symile-MIMIC**.

### Fuente de las imágenes
Las imágenes CXR **no se leen desde disco como JPGs sueltos**, sino desde los arrays pre-procesados `cxr_train.npy / cxr_val.npy / cxr_test.npy` incluidos en el dataset.  
El CSV (`train_clean.csv`, etc.) contiene la columna `cxr_path` con rutas tipo `files/p17/.../imagen.jpg` — estas rutas sirven como **identificador** para hacer el join por `hadm_id` con el npy, pero las imágenes ya están cargadas como arrays en el npy.

> ⚠️ **Nota sobre tamaño de test**: el test set fue reducido a 1/10 del original en la limpieza de datos (464 registros en el CSV vs. 4640 en el npy original). El código maneja esto haciendo join por `hadm_id` entre el CSV limpio y el npy, tomando solo las filas que coinciden.

### Arquitectura
```
cxr_train.npy  →  [N, C, H, W]  (imágenes pre-procesadas)
      │
  ResNet-50 (pesos CheXpert, torchxrayvision)
      │
   GAP → [B, 2048]
      │
  ┌───┴────────────────────────┐
  Proyección [2048→512]    Rama metadatos [13→64]
  └───────────┬────────────────┘
         Concatenación [576]
              │
         Head lineal → [B, 6]  logits
              │
    Módulo correlación 6×6 (aprendible, skip connection)
              │
           Sigmoid → probabilidades
```

### Fixes aplicados (todos marcados con [FIX N]):
- **[FIX 1]** Imágenes leídas desde `.npy` en lugar de JPGs en disco
- **[FIX 2]** Join por `hadm_id` para alinear CSV y npy (maneja el 1/10 del test)
- **[FIX 3]** `ReduceLROnPlateau`: eliminado `verbose=False` (deprecado PyTorch ≥2.2)
- **[FIX 4]** `build_optimizer_and_scheduler` acepta `num_epochs_override` para reconstrucción post-unfreeze sin que `OneCycleLR` exceda pasos
- **[FIX 5]** Guardia NaN en AUC del loop interno (folds con pocos positivos)
- **[FIX 6]** `gc.collect()` + `cuda.empty_cache()` combinados para liberar memoria
- **[FIX 7]** `auc_macros` / `f1_macros` recalculadas al inicio de celda 12 (scope Jupyter)
- **[FIX 8]** Barra de progreso `tqdm` en entrenamiento y evaluación


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 1: INSTALACIÓN DE DEPENDENCIAS
# ══════════════════════════════════════════════════════════════════════════════

import subprocess, sys

packages = [
    "torchxrayvision",   # Pesos CheXpert + utilidades CXR
    "albumentations",    # Augmentación de imagen avanzada
    "scikit-learn",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "tqdm",              # [FIX 8] barras de progreso
    "Pillow",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

print("✅ Dependencias instaladas.")


✅ Dependencias instaladas.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 2: IMPORTACIONES Y SEMILLAS
# ══════════════════════════════════════════════════════════════════════════════

import os, gc, warnings, random, json, copy, time
from pathlib import Path
from itertools import product as itertools_product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm          # [FIX 8] tqdm con soporte notebook

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as tv_models

try:
    import torchxrayvision as xrv
    XRV_AVAILABLE = True
except ImportError:
    XRV_AVAILABLE = False
    print("⚠ torchxrayvision no disponible — se usará ResNet-50 ImageNet como fallback.")

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

warnings.filterwarnings("ignore")

# ── Reproducibilidad ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Dispositivo : {DEVICE}")
print(f"   PyTorch     : {torch.__version__}")
print(f"   XRV         : {'disponible' if XRV_AVAILABLE else 'NO disponible (fallback ImageNet)'}")


✅ Dispositivo : cpu
   PyTorch     : 2.8.0+cpu
   XRV         : disponible


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 3: CONFIGURACIÓN DE RUTAS Y CONSTANTES GLOBALES
# ══════════════════════════════════════════════════════════════════════════════
# [FIX 1] Las imágenes se leen desde archivos .npy, NO desde JPGs en disco.
#         cxr_path en el CSV es solo un identificador; la alineación real
#         se hace por hadm_id (ver Celda 6).
# ══════════════════════════════════════════════════════════════════════════════

BASE_DATA_DIR = Path(
    r"C:\TFM\1.OPCIÓN - SYMILE MIMIC\SYMILE-MIMIC-A-MULTIMODAL-CLINICAL-DATASET-OF-CHEST-X-RAYS-ELECTROCARDIOGRAMS-AND-BLOOD-LABS-FROM-MIMIC-IV-1.0.0"
)

# ── CSVs limpios ──────────────────────────────────────────────────────────────
CSV_DIR   = BASE_DATA_DIR / "data_csv" / "clean"
TRAIN_CSV = CSV_DIR / "train_clean.csv"
VAL_CSV   = CSV_DIR / "val_clean.csv"
TEST_CSV  = CSV_DIR / "test_clean.csv"

# ── Arrays .npy de imágenes CXR ──────────────────────────────────────────────
# [FIX 1] Fuente real de las imágenes: arrays pre-procesados por el dataset.
# Shape esperado: (N, C, H, W) o (N, H, W) según el pre-procesado de Symile-MIMIC.
NPY_DIR       = BASE_DATA_DIR / "data_npy"
CXR_TRAIN_NPY = NPY_DIR / "train" / "cxr_train.npy"
CXR_VAL_NPY   = NPY_DIR / "val"   / "cxr_val.npy"
CXR_TEST_NPY  = NPY_DIR / "test"  / "cxr_test.npy"

# hadm_id arrays — necesarios para el join CSV ↔ npy [FIX 2]
HADM_TRAIN_NPY = NPY_DIR / "train" / "hadm_id_train.npy"
HADM_VAL_NPY   = NPY_DIR / "val"   / "hadm_id_val.npy"
HADM_TEST_NPY  = NPY_DIR / "test"  / "hadm_id_test.npy"

# ── Outputs ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("outputs_resnet50_cxr")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Etiquetas multilabel ──────────────────────────────────────────────────────
LABELS   = ["Atelectasis", "Cardiomegaly", "Edema", "Lung Opacity", "No Finding", "Pleural Effusion"]
N_LABELS = len(LABELS)

# ── Pesos positivos por etiqueta derivados del EDA (ratio neg/pos observados) ─
POS_WEIGHTS = {
    "Atelectasis"    : 0.04,   # 24:1
    "Cardiomegaly"   : 0.30,   # 4:1
    "Edema"          : 0.90,   # ~1.2:1
    "Lung Opacity"   : 0.10,   # 18:1
    "No Finding"     : 1.00,   # solo positivos
    "Pleural Effusion": 0.50,  # 2:1
}

# ── Parámetros de imagen ──────────────────────────────────────────────────────
IMG_SIZE       = 224
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD  = [0.229, 0.224, 0.225]

# ── Metadatos clínicos ────────────────────────────────────────────────────────
GENDER_MAP    = {0: 0, 1: 1}
RACE_MAP      = {"UNKNOWN": 0, "WHITE": 1, "BLACK": 2, "ASIAN": 3, "HISPANIC_LATINO": 4}
ADMISSION_MAP = {"SCHEDULED": 0, "EMERGENCY": 1, "OBSERVATION": 2, "URGENT": 3}
CXR_VIEW_MAP  = {"AP": 0, "PA": 1}
META_DIM      = 13   # edad_norm + gender + 5 race + 4 admission + 2 view
META_EMBED    = 64
IMG_PROJ      = 512

print("✅ Constantes y rutas configuradas.")
print(f"   CXR train npy : {CXR_TRAIN_NPY}")
print(f"   CXR val npy   : {CXR_VAL_NPY}")
print(f"   CXR test npy  : {CXR_TEST_NPY}")


✅ Constantes y rutas configuradas.
   CXR train npy : C:\TFM\1.OPCIÓN - SYMILE MIMIC\SYMILE-MIMIC-A-MULTIMODAL-CLINICAL-DATASET-OF-CHEST-X-RAYS-ELECTROCARDIOGRAMS-AND-BLOOD-LABS-FROM-MIMIC-IV-1.0.0\data_npy\train\cxr_train.npy
   CXR val npy   : C:\TFM\1.OPCIÓN - SYMILE MIMIC\SYMILE-MIMIC-A-MULTIMODAL-CLINICAL-DATASET-OF-CHEST-X-RAYS-ELECTROCARDIOGRAMS-AND-BLOOD-LABS-FROM-MIMIC-IV-1.0.0\data_npy\val\cxr_val.npy
   CXR test npy  : C:\TFM\1.OPCIÓN - SYMILE MIMIC\SYMILE-MIMIC-A-MULTIMODAL-CLINICAL-DATASET-OF-CHEST-X-RAYS-ELECTROCARDIOGRAMS-AND-BLOOD-LABS-FROM-MIMIC-IV-1.0.0\data_npy\test\cxr_test.npy


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 4: CARGA DE DATOS Y ALINEACIÓN CSV ↔ NPY
# ══════════════════════════════════════════════════════════════════════════════
# [FIX 1] Las imágenes están en cxr_train.npy / cxr_val.npy / cxr_test.npy,
#         NO como JPGs en disco. cxr_path en el CSV es solo un identificador.
#
# [FIX 2] El join se hace por hadm_id:
#   - hadm_id_train.npy contiene los hadm_id en el mismo orden que cxr_train.npy
#   - train_clean.csv tiene sus propios hadm_id (subconjunto limpio)
#   - Construimos un índice npy_hadm → posición_en_npy y luego filtramos
#     solo las filas del CSV que tienen hadm_id en el npy.
#
# [FIX 2b] El test set fue reducido a 1/10 (464 filas en CSV vs 4640 en npy).
#          El join por hadm_id resuelve esto automáticamente: solo se toman
#          las 464 filas del CSV que coinciden con sus posiciones en el npy.
# ══════════════════════════════════════════════════════════════════════════════

def load_split(csv_path, cxr_npy_path, hadm_npy_path, split_name="train"):
    """
    Carga un split completo alineando CSV y npy por hadm_id.

    Returns:
        df       : DataFrame con las filas del CSV que tienen imagen en el npy.
        cxr_npy  : Array numpy con TODAS las imágenes del npy (shape [N, ...]).
        hadm_to_npy_idx : dict {hadm_id: índice_en_npy} para lookup rápido.
    """
    print(f"\n── Cargando split '{split_name}' ────────────────────────────────────")

    # CSV limpio
    df = pd.read_csv(csv_path, sep=";")
    print(f"   CSV filas          : {len(df):,}")

    # hadm_id del npy (alineados con el array de imágenes)
    hadm_npy = np.load(hadm_npy_path, allow_pickle=True)
    print(f"   npy hadm_id count  : {len(hadm_npy):,}")

    # Construir índice hadm → posición en npy
    hadm_to_npy_idx = {int(h): i for i, h in enumerate(hadm_npy)}

    # Filtrar CSV a filas con imagen disponible en el npy
    df["_npy_idx"] = df["hadm_id"].apply(lambda h: hadm_to_npy_idx.get(int(h), -1))
    n_before = len(df)
    df = df[df["_npy_idx"] >= 0].reset_index(drop=True)
    print(f"   Filas con imagen   : {len(df):,}  (descartadas: {n_before - len(df)})")

    # Cargar el array de imágenes completo en memoria
    # Shape típico Symile-MIMIC: (N, 1, 224, 224) float32 en rango [-1024, 1024]
    # o (N, 224, 224). Lo normalizamos en el Dataset.
    print(f"   Cargando {cxr_npy_path.name} ...", end=" ", flush=True)

    # IMPORTANTE:
    # mmap_mode="r" evita cargar los ~11 GB completos en RAM.
    # El array se accede directamente desde disco cuando se necesita.
    cxr_npy = np.load(
        cxr_npy_path,
        mmap_mode="r"
    )

    print(f"shape={cxr_npy.shape}  dtype={cxr_npy.dtype}")

    return df, cxr_npy, hadm_to_npy_idx


# Carga de los tres splits
df_train, cxr_train_npy, hadm_to_npy_train = load_split(TRAIN_CSV, CXR_TRAIN_NPY, HADM_TRAIN_NPY, "train")
df_val,   cxr_val_npy,   hadm_to_npy_val   = load_split(VAL_CSV,   CXR_VAL_NPY,   HADM_VAL_NPY,   "val")
df_test,  cxr_test_npy,  hadm_to_npy_test  = load_split(TEST_CSV,  CXR_TEST_NPY,  HADM_TEST_NPY,  "test")

print(f"\n✅ Datos cargados:")
print(f"   Train : {len(df_train):,} muestras  |  CXR npy shape: {cxr_train_npy.shape}")
print(f"   Val   : {len(df_val):,}  muestras  |  CXR npy shape: {cxr_val_npy.shape}")
print(f"   Test  : {len(df_test):,}   muestras  |  CXR npy shape: {cxr_test_npy.shape}")
print(f"   [FIX 2b] Test CSV=464 filas alineadas con npy={cxr_test_npy.shape[0]} entradas totales → join correcto.")

# ── Inspección rápida del rango de valores del npy ────────────────────────────
sample = cxr_train_npy[0]
print(f"\n   Muestra npy[0]: shape={sample.shape}  min={sample.min():.1f}  max={sample.max():.1f}  mean={sample.mean():.1f}")



── Cargando split 'train' ────────────────────────────────────
   CSV filas          : 10,000
   npy hadm_id count  : 10,000
   Filas con imagen   : 10,000  (descartadas: 0)
   Cargando cxr_train.npy ... shape=(10000, 3, 320, 320)  dtype=float32

── Cargando split 'val' ────────────────────────────────────
   CSV filas          : 750
   npy hadm_id count  : 750
   Filas con imagen   : 750  (descartadas: 0)
   Cargando cxr_val.npy ... shape=(750, 3, 320, 320)  dtype=float32

── Cargando split 'test' ────────────────────────────────────
   CSV filas          : 464
   npy hadm_id count  : 4,640
   Filas con imagen   : 464  (descartadas: 0)
   Cargando cxr_test.npy ... shape=(4640, 3, 320, 320)  dtype=float32

✅ Datos cargados:
   Train : 10,000 muestras  |  CXR npy shape: (10000, 3, 320, 320)
   Val   : 750  muestras  |  CXR npy shape: (750, 3, 320, 320)
   Test  : 464   muestras  |  CXR npy shape: (4640, 3, 320, 320)
   [FIX 2b] Test CSV=464 filas alineadas con npy=4640 entradas totales

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 5: GRID DE HIPERPARÁMETROS — VERSIÓN CPU OPTIMIZADA
# ══════════════════════════════════════════════════════════════════════════════
# Coste original v2:  3 × 6  × 3 = 54  entrenamientos internos + 3 finales
# Coste reducido CPU: 3 × 3  × 3 = 27  entrenamientos internos + 3 finales
# Reducción: ~2× menos entrenamientos + épocas 6→4 + early stopping 2→1
# = ~3×-4× menos tiempo total respecto a la versión anterior.
#
# Cambios respecto al grid v2:
#   N_RANDOM_CONFIGS : 6 → 3    mitad de configs a evaluar
#   num_epochs       : [6] → [4] 2 épocas menos por entrenamiento
#   dropout_rate     : [0.3, 0.5] → [0.4]  un valor central, menos combinaciones
#   weight_decay     : [1e-4, 1e-3] → [3e-4] un valor central
#   uncertainty_policy: [zeros, ones] → [zeros]  la más común en CheXpert
# ══════════════════════════════════════════════════════════════════════════════

HYPERPARAM_GRID = {
    "lr_backbone"          : [1e-5, 3e-5],
    "lr_head"              : [1e-4, 3e-4],
    "scheduler"            : ["cosine"],
    "num_epochs"           : [4],
    "unfreeze_epoch"       : [2],
    "unfreeze_layers"      : ["layer4"],
    "uncertainty_policy"   : ["zeros"],
    "dropout_rate"         : [0.4],
    "weight_decay"         : [3e-4],
    "batch_size"           : [32],
    "use_label_correlation": [True, False],
    "use_meta_branch"      : [True],
    "threshold_search"     : [True],
    "augmentation_level"   : ["moderate"],
}

N_RANDOM_CONFIGS = 3

all_keys   = list(HYPERPARAM_GRID.keys())
all_values = list(HYPERPARAM_GRID.values())
all_combos = list(itertools_product(*all_values))
np.random.shuffle(all_combos)
SAMPLED_CONFIGS = [dict(zip(all_keys, c)) for c in all_combos[:N_RANDOM_CONFIGS]]

print(f"✅ Grid reducido para CPU.")
print(f"   Combinaciones posibles : {len(all_combos):,}")
print(f"   Configs muestreadas    : {N_RANDOM_CONFIGS}")
print(f"   Entrenamientos totales : 3 folds × {N_RANDOM_CONFIGS} configs × 3 inner = "
      f"{3 * N_RANDOM_CONFIGS * 3} internos + 3 finales")
print(f"   (vs 54 internos de la versión anterior)")

✅ Grid reducido para CPU.
   Combinaciones posibles : 8
   Configs muestreadas    : 3
   Entrenamientos totales : 3 folds × 3 configs × 3 inner = 27 internos + 3 finales
   (vs 54 internos de la versión anterior)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 6: PIPELINES DE AUGMENTACIÓN
# ══════════════════════════════════════════════════════════════════════════════
# [FIX 9] Las imágenes del npy vienen con shape (3, 320, 320) float32 ya
#         normalizadas (~[-2.1, 2.6], convenio ImageNet). Por tanto:
#   - NO convertir a uint8 ni re-normalizar (destruiría la normalización).
#   - Albumentations recibe HxWxC float32 en ese rango directamente.
#   - A.Normalize se OMITE del pipeline (ya está hecha).
#   - A.Resize se OMITE en test/val (ya son 320×320); en train se puede
#     aplicar crop/flip sobre el tamaño nativo.
# ══════════════════════════════════════════════════════════════════════════════

def get_augmentation_pipeline(level: str, img_size: int = IMG_SIZE):
    """
    Devuelve un pipeline albumentations adaptado a imágenes float32
    ya normalizadas con shape (3, H, W) → se transpone a (H, W, 3) en el Dataset.
    NO incluye A.Normalize (las imágenes ya están normalizadas).
    """
    to_tensor = ToTensorV2()  # HxWxC → CxHxW, mantiene dtype float32

    if level == "test":
        # Sin augmentación: solo asegurar tamaño correcto y convertir a tensor
        return A.Compose([
            A.Resize(img_size, img_size),
            to_tensor,
        ])

    elif level == "moderate":
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=10, p=0.5),
            # BrightnessContrast y ShiftScaleRotate funcionan sobre float
            # con is_check_shapes=False para evitar warnings de albumentations
            A.RandomBrightnessContrast(
                brightness_limit=0.15, contrast_limit=0.15, p=0.4
            ),
            A.ShiftScaleRotate(
                shift_limit=0.05, scale_limit=0.05, rotate_limit=0, p=0.3
            ),
            to_tensor,
        ])

    elif level == "aggressive":
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.07, scale_limit=0.08, rotate_limit=10, p=0.4
            ),
            A.ElasticTransform(alpha=80, sigma=8, p=0.3),
            A.GridDistortion(num_steps=5, distort_limit=0.15, p=0.2),
            A.RandomBrightnessContrast(
                brightness_limit=0.25, contrast_limit=0.25, p=0.5
            ),
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),
            A.GaussNoise(var_limit=(0.001, 0.005), p=0.3),  # var pequeña: imagen ya en [-2,2]
            A.OneOf([
                A.GaussianBlur(blur_limit=3),
                A.MotionBlur(blur_limit=3),
            ], p=0.2),
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
            A.CoarseDropout(
                max_holes=8, max_height=32, max_width=32, fill_value=0.0, p=0.2
            ),
            to_tensor,
        ])

    else:
        raise ValueError(f"Nivel desconocido: '{level}'. Usa 'moderate', 'aggressive' o 'test'.")


print("✅ Pipelines de augmentación definidos (float32 nativo, sin re-normalización).")
print("   [FIX 9] A.Normalize eliminado — imágenes ya normalizadas en el npy.")

✅ Pipelines de augmentación definidos (float32 nativo, sin re-normalización).
   [FIX 9] A.Normalize eliminado — imágenes ya normalizadas en el npy.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 7: DATASET PYTORCH — CXR MULTILABEL DESDE NPY
# ══════════════════════════════════════════════════════════════════════════════
# [FIX 9] Las imágenes ya vienen como float32 (3, 320, 320) normalizadas.
#         npy_to_hwc_float: solo transpone CxHxW → HxWxC para albumentations.
#         NO hay conversión a uint8 ni normalización adicional.
# ══════════════════════════════════════════════════════════════════════════════

def npy_to_hwc_float(arr: np.ndarray) -> np.ndarray:
    """
    Convierte un array de imagen del npy a float32 HxWxC listo para albumentations.
    Soporta:
      (C, H, W)  → transpone a (H, W, C)   [caso del npy: (3, 320, 320)]
      (H, W)     → replica a  (H, W, 3)
      (H, W, C)  → devuelve tal cual
    No normaliza: las imágenes ya vienen normalizadas del npy.
    """
    arr = arr.astype(np.float32)

    if arr.ndim == 3 and arr.shape[0] in (1, 3):
        # (C, H, W) → (H, W, C)
        arr = arr.transpose(1, 2, 0)
    elif arr.ndim == 2:
        # (H, W) → (H, W, 3)
        arr = np.stack([arr, arr, arr], axis=-1)
    # Si ya es (H, W, C) no hace falta nada

    # Si es monocanal (H, W, 1) → replicar a (H, W, 3) para el backbone RGB
    if arr.shape[2] == 1:
        arr = np.repeat(arr, 3, axis=2)

    return arr   # float32, HxWxC, valores en [-2.1, 2.6]


class CXRMultilabelDataset(Dataset):
    """
    Dataset PyTorch para radiografías de tórax multilabel.
    [FIX 1] Lee imágenes desde npy (no JPGs en disco).
    [FIX 9] Usa npy_to_hwc_float: no re-normaliza, respeta float32 nativo.
    """

    def __init__(self, dataframe, cxr_npy, augmentation_pipeline,
                 uncertainty_policy="zeros", labels=LABELS):
        self.df                 = dataframe.reset_index(drop=True)
        self.cxr_npy            = cxr_npy
        self.transform          = augmentation_pipeline
        self.uncertainty_policy = uncertainty_policy
        self.labels             = labels

        age_col      = self.df["age"].astype(float)
        self.age_min = age_col.min()
        self.age_max = age_col.max()

    def __len__(self):
        return len(self.df)

    def _encode_labels_and_mask(self, row):
        label_vec = np.zeros(len(self.labels), dtype=np.float32)
        mask_vec  = np.zeros(len(self.labels), dtype=np.float32)
        for i, lbl in enumerate(self.labels):
            val = row[lbl]
            if pd.isna(val):
                label_vec[i], mask_vec[i] = 0.0, 0.0
            elif val == -1:
                mask_vec[i]  = 1.0
                label_vec[i] = 1.0 if self.uncertainty_policy == "ones" else 0.0
            else:
                label_vec[i] = float(val)
                mask_vec[i]  = 1.0
        return label_vec, mask_vec

    def _encode_metadata(self, row):
        age_norm = (float(row["age"]) - self.age_min) / (self.age_max - self.age_min + 1e-8)
        gender   = float(GENDER_MAP.get(row["gender"], 0))
        race_vec = np.zeros(len(RACE_MAP),      dtype=np.float32)
        adm_vec  = np.zeros(len(ADMISSION_MAP), dtype=np.float32)
        view_vec = np.zeros(len(CXR_VIEW_MAP),  dtype=np.float32)
        race_vec[RACE_MAP.get(str(row["race"]), 0)]               = 1.0
        adm_vec[ADMISSION_MAP.get(str(row["admission_type"]), 1)] = 1.0
        view_vec[CXR_VIEW_MAP.get(str(row["cxr_view"]), 0)]      = 1.0
        return np.concatenate([[age_norm, gender], race_vec, adm_vec, view_vec])

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # [FIX 1] Leer desde npy con el índice pre-calculado
        npy_idx = int(row["_npy_idx"])
        img_arr = self.cxr_npy[npy_idx]          # (3, 320, 320) float32

        # [FIX 9] Transponer a HxWxC para albumentations, sin re-normalizar
        image = npy_to_hwc_float(img_arr)         # (320, 320, 3) float32

        image_tensor = self.transform(image=image)["image"].float()  # (3, H, W)

        labels, mask = self._encode_labels_and_mask(row)
        metadata     = self._encode_metadata(row)

        return {
            "image"   : image_tensor,
            "labels"  : torch.tensor(labels,   dtype=torch.float32),
            "mask"    : torch.tensor(mask,      dtype=torch.float32),
            "metadata": torch.tensor(metadata,  dtype=torch.float32),
            "hadm_id" : int(row["hadm_id"]),
        }


print("✅ CXRMultilabelDataset definido.")
print("   [FIX 9] npy_to_hwc_float: transpone (3,320,320) → (320,320,3), sin re-normalizar.")

✅ CXRMultilabelDataset definido.
   [FIX 9] npy_to_hwc_float: transpone (3,320,320) → (320,320,3), sin re-normalizar.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 8: MASKED BINARY CROSS-ENTROPY (CONVENIO CHEXPERT)
# ══════════════════════════════════════════════════════════════════════════════
# mask=1 → etiqueta observable → contribuye al gradiente.
# mask=0 → etiqueta NaN       → excluida completamente.
# pos_weight compensa el desequilibrio de clases por etiqueta.
# ══════════════════════════════════════════════════════════════════════════════

class MaskedBCELoss(nn.Module):
    def __init__(self, pos_weights_dict=None, labels=LABELS):
        super().__init__()
        if pos_weights_dict is not None:
            w = torch.tensor([pos_weights_dict.get(l, 1.0) for l in labels], dtype=torch.float32)
        else:
            w = torch.ones(len(labels), dtype=torch.float32)
        self.register_buffer("pos_weights", w)

    def forward(self, logits, labels, mask):
        bce = F.binary_cross_entropy_with_logits(
            logits, labels, pos_weight=self.pos_weights.to(logits.device), reduction="none"
        )
        masked = bce * mask
        return masked.sum() / mask.sum().clamp(min=1e-8)


criterion = MaskedBCELoss(pos_weights_dict=POS_WEIGHTS).to(DEVICE)
print("✅ MaskedBCELoss lista con pos_weights del EDA.")


✅ MaskedBCELoss lista con pos_weights del EDA.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 9: ARQUITECTURA — ResNet-50 + Correlación + Metadatos
# [FIX 10] Colapso de 3→1 canal para backbone CheXpert en forward()
# [FIX 11] Detección DINÁMICA de la dimensión de salida del backbone XRV:
#          torchxrayvision ResNet no sigue la estructura estándar de torchvision
#          y puede tener pooling interno propio. Se hace un forward con un
#          tensor dummy para medir la dimensión real antes de construir img_proj.
# ══════════════════════════════════════════════════════════════════════════════

class LabelCorrelationModule(nn.Module):
    def __init__(self, n_labels=N_LABELS):
        super().__init__()
        self.correlation = nn.Linear(n_labels, n_labels, bias=False)
        nn.init.eye_(self.correlation.weight)
        self.correlation.weight.data *= 0.1
        self.norm = nn.LayerNorm(n_labels)

    def forward(self, logits):
        return self.norm(logits + self.correlation(logits))


class MetadataBranch(nn.Module):
    def __init__(self, meta_dim=META_DIM, meta_embed=META_EMBED):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(meta_dim, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True), nn.Dropout(0.2),
            nn.Linear(128, meta_embed), nn.BatchNorm1d(meta_embed), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)


def _infer_backbone_out_dim(backbone, img_size=IMG_SIZE):
    """
    [FIX 11] Pasa un tensor dummy 1-canal por el backbone + GAP para medir
    la dimensión de salida real. Necesario porque torchxrayvision ResNet
    tiene arquitectura interna distinta a torchvision y puede incluir
    su propio pooling, produciendo shapes inesperados.
    """
    was_training = backbone.training
    backbone.eval()
    with torch.no_grad():
        dummy = torch.zeros(2, 1, img_size, img_size)  # 1 canal, batch=2
        try:
            out = backbone(dummy)
        except Exception:
            # Algunos modelos XRV devuelven un dict o tienen capas adicionales
            # En ese caso extraemos el atributo .model interno
            raise RuntimeError(
                "El backbone no acepta input directo. "
                "Verifica la estructura de torchxrayvision."
            )
        # Si ya tiene pooling interno: shape puede ser (B, C) o (B, C, 1, 1)
        if out.ndim == 4:
            gap = nn.AdaptiveAvgPool2d(1)
            out = gap(out).flatten(1)
        elif out.ndim == 3:
            out = out.mean(dim=-1)  # (B, C, L) → (B, C)
        elif out.ndim == 2:
            pass  # ya es (B, C)
        dim = out.shape[1]
    if was_training:
        backbone.train()
    return dim, (out.ndim != 2)  # dim, necesita_gap_externo


class CXRResNet50(nn.Module):
    def __init__(self, n_labels=N_LABELS, dropout_rate=0.5,
                 use_label_correlation=True, use_meta_branch=True,
                 pretrained_source="chexpert"):
        super().__init__()
        self.use_label_correlation = use_label_correlation
        self.use_meta_branch       = use_meta_branch

        # ── Backbone ──────────────────────────────────────────────────────────
        if pretrained_source == "chexpert" and XRV_AVAILABLE:
            try:
                # DESPUÉS [FIX 12]: extraer solo el backbone sin la cabeza clasificadora XRV.
                # xrv_model.model es el ResNet completo de torchvision internamente.
                # Quitamos los últimos 2 módulos (avgpool + fc) para quedarnos con
                # el extractor de features puro que sí da (B, 2048, H, W).
                xrv_model     = xrv.models.ResNet(weights="resnet50-res512-all")
                full           = xrv_model.model
                self.backbone  = nn.Sequential(*list(full.children())[:-2])
                print("   ✓ Backbone: ResNet-50 pesos CheXpert (torchxrayvision)")
                print("   ✓ Backbone: ResNet-50 pesos CheXpert (torchxrayvision)")
            except Exception as e:
                print(f"   ⚠ XRV falló ({e}) → ResNet-50 ImageNet")
                backbone      = tv_models.resnet50(weights="IMAGENET1K_V1")
                self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        else:
            backbone      = tv_models.resnet50(weights="IMAGENET1K_V1")
            self.backbone = nn.Sequential(*list(backbone.children())[:-2])
            print("   ✓ Backbone: ResNet-50 ImageNet")

        # [FIX 11] Detectar dimensión de salida dinámicamente
        backbone_dim, self._needs_gap = _infer_backbone_out_dim(self.backbone, IMG_SIZE)
        print(f"   ✓ Backbone out dim detectada: {backbone_dim}  |  needs_GAP: {self._needs_gap}")

        self.gap      = nn.AdaptiveAvgPool2d(1)  # solo se usa si _needs_gap=True
        self.img_proj = nn.Sequential(
            nn.Linear(backbone_dim, IMG_PROJ),
            nn.BatchNorm1d(IMG_PROJ),
            nn.ReLU(inplace=True),
        )

        # ── Rama metadatos ────────────────────────────────────────────────────
        if use_meta_branch:
            self.meta_branch = MetadataBranch()
            fusion_dim = IMG_PROJ + META_EMBED
        else:
            self.meta_branch = None
            fusion_dim = IMG_PROJ

        # ── Cabeza clasificadora ──────────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(fusion_dim, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(256, n_labels),
        )
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

        # ── Módulo correlación ────────────────────────────────────────────────
        self.label_corr = LabelCorrelationModule(n_labels) if use_label_correlation else None

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_layers(self, strategy="layer4"):
        for p in self.backbone.parameters():
            p.requires_grad = False
        if strategy == "all":
            for p in self.backbone.parameters():
                p.requires_grad = True
            print("   🔓 Backbone completo descongelado.")
        elif strategy == "layer4":
            for n, p in self.backbone.named_parameters():
                if "layer4" in n: p.requires_grad = True
            print("   🔓 layer4 descongelada.")
        elif strategy == "layer3_4":
            for n, p in self.backbone.named_parameters():
                if "layer3" in n or "layer4" in n: p.requires_grad = True
            print("   🔓 layer3+layer4 descongeladas.")

    def forward(self, image, metadata=None):
        # [FIX 10] Colapsar 3→1 canal para backbone CheXpert (espera grayscale)
        if image.shape[1] == 3:
            image = image.mean(dim=1, keepdim=True)   # (B,3,H,W) → (B,1,H,W)

        out = self.backbone(image)

        # [FIX 11] Adaptar shape de salida del backbone
        if out.ndim == 4:
            feat = self.gap(out).flatten(1)   # (B,C,H,W) → (B,C)
        elif out.ndim == 3:
            feat = out.mean(dim=-1)           # (B,C,L)   → (B,C)
        else:
            feat = out                        # (B,C) ya listo

        proj = self.img_proj(feat)

        if self.use_meta_branch and metadata is not None:
            fused = torch.cat([proj, self.meta_branch(metadata)], dim=1)
        else:
            fused = proj

        logits = self.head(fused)
        if self.label_corr is not None:
            logits = self.label_corr(logits)
        return {"logits": logits, "probs": torch.sigmoid(logits)}


print("✅ CXRResNet50 definida con detección dinámica de dimensión.")
print("   [FIX 10] 3→1 canal en forward para backbone CheXpert")
print("   [FIX 11] _infer_backbone_out_dim detecta shape real antes de construir img_proj")

✅ CXRResNet50 definida con detección dinámica de dimensión.
   [FIX 10] 3→1 canal en forward para backbone CheXpert
   [FIX 11] _infer_backbone_out_dim detecta shape real antes de construir img_proj


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 10: UTILIDADES — MÉTRICAS, UMBRALES, OPTIMIZADOR
# [FIX 3] ReduceLROnPlateau: eliminado verbose=False (deprecado PyTorch ≥2.2)
# [FIX 4] build_optimizer_and_scheduler acepta num_epochs_override para evitar
#         que OneCycleLR exceda pasos al reconstruir tras descongelar backbone
# [FIX 5] macro_AUC con lista vacía devuelve nan en vez de crashear
# ══════════════════════════════════════════════════════════════════════════════

def compute_multilabel_metrics(all_probs, all_labels, all_masks,
                                thresholds=None, labels=LABELS):
    if thresholds is None:
        thresholds = np.full(len(labels), 0.5)
    metrics, auc_list, ap_list, f1_list = {}, [], [], []
    for i, lbl in enumerate(labels):
        vm     = all_masks[:, i] == 1
        y_true = all_labels[vm, i]
        y_prob = all_probs[vm, i]
        y_pred = (y_prob >= thresholds[i]).astype(float)
        n_pos  = int(y_true.sum())
        n_neg  = int((1 - y_true).sum())
        if n_pos < 2 or n_neg < 2:
            auc, ap = float("nan"), float("nan")
        else:
            auc = roc_auc_score(y_true, y_prob)
            ap  = average_precision_score(y_true, y_prob)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        metrics[lbl] = {"AUC": auc, "AP": ap, "F1": f1, "n_pos": n_pos, "n_neg": n_neg}
        if not np.isnan(auc):
            auc_list.append(auc); ap_list.append(ap)
        f1_list.append(f1)
    # [FIX 5]
    metrics["macro_AUC"] = float(np.nanmean(auc_list)) if auc_list else float("nan")
    metrics["macro_AP"]  = float(np.nanmean(ap_list))  if ap_list  else float("nan")
    metrics["macro_F1"]  = float(np.nanmean(f1_list))  if f1_list  else float("nan")
    return metrics


def find_optimal_thresholds(all_probs, all_labels, all_masks,
                             labels=LABELS, n_thresholds=50):
    thresholds = np.full(len(labels), 0.5)
    for i, lbl in enumerate(labels):
        vm     = all_masks[:, i] == 1
        y_true = all_labels[vm, i]
        y_prob = all_probs[vm, i]
        if y_true.sum() < 2:
            continue
        best_f1, best_thr = -1.0, 0.5
        for thr in np.linspace(0.1, 0.9, n_thresholds):
            f1 = f1_score(y_true, (y_prob >= thr).astype(float), zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr = f1, thr
        thresholds[i] = best_thr
    return thresholds


def build_optimizer_and_scheduler(model, config, train_loader_len,
                                   num_epochs_override=None):  # [FIX 4]
    """
    num_epochs_override: épocas efectivas para el scheduler.
    Usar al reconstruir tras unfreeze para pasar épocas RESTANTES.
    """
    num_epochs = num_epochs_override if num_epochs_override is not None else config["num_epochs"]

    head_params = (
        list(model.img_proj.parameters())
        + list(model.head.parameters())
        + (list(model.label_corr.parameters())  if model.label_corr  else [])
        + (list(model.meta_branch.parameters()) if model.meta_branch else [])
    )
    param_groups = [
        {"params": model.backbone.parameters(), "lr": config["lr_backbone"], "name": "backbone"},
        {"params": head_params,                 "lr": config["lr_head"],     "name": "head"},
    ]
    optimizer = torch.optim.AdamW(param_groups, weight_decay=config["weight_decay"])

    sched = config["scheduler"]
    if sched == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs, eta_min=1e-7)
    elif sched == "onecycle":
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=[config["lr_backbone"] * 10, config["lr_head"] * 10],
            steps_per_epoch=train_loader_len,
            epochs=num_epochs, pct_start=0.1)
    elif sched == "plateau":
        # [FIX 3] verbose=False eliminado — deprecado en PyTorch ≥2.2
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3)
    else:
        scheduler = None
    return optimizer, scheduler


print("✅ Utilidades definidas.")
print("   [FIX 3] ReduceLROnPlateau sin verbose=False")
print("   [FIX 4] build_optimizer_and_scheduler con num_epochs_override")
print("   [FIX 5] macro_AUC seguro ante listas vacías")


✅ Utilidades definidas.
   [FIX 3] ReduceLROnPlateau sin verbose=False
   [FIX 4] build_optimizer_and_scheduler con num_epochs_override
   [FIX 5] macro_AUC seguro ante listas vacías


In [ ]:
def train_model(model, df_train_fold, df_val_fold, cxr_npy_train, cxr_npy_val,
                config, verbose=True):
    aug_train = get_augmentation_pipeline(config["augmentation_level"])
    aug_val   = get_augmentation_pipeline("test")

    ds_train = CXRMultilabelDataset(df_train_fold, cxr_npy_train, aug_train,
                                     uncertainty_policy=config["uncertainty_policy"])
    ds_val   = CXRMultilabelDataset(df_val_fold,   cxr_npy_val,   aug_val,
                                     uncertainty_policy=config["uncertainty_policy"])

    # [CPU-FIX] num_workers>0 paralleliza la carga mientras la CPU entrena.
    # En Windows usa num_workers=2 con multiprocessing_context="spawn".
    # En Linux/Mac puedes subir a 4.
    N_WORKERS = 2

    loader_train = DataLoader(
        ds_train,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=N_WORKERS,
        pin_memory=False,          # False en CPU (no hay GPU)
        drop_last=True,
        persistent_workers=True,   # reutiliza workers entre épocas
        prefetch_factor=2,         # precarga 2 batches por worker
    )
    loader_val = DataLoader(
        ds_val,
        batch_size=config["batch_size"] * 2,
        shuffle=False,
        num_workers=N_WORKERS,
        pin_memory=False,
        persistent_workers=True,
        prefetch_factor=2,
    )

    optimizer, scheduler = build_optimizer_and_scheduler(model, config, len(loader_train))
    model.freeze_backbone()

    # [CPU-FIX] torch.compile acelera ~15-25% en CPU (PyTorch ≥2.0)
    try:
        model = torch.compile(model)
    except Exception:
        pass  # Si falla (PyTorch <2.0 o Windows), continúa sin compilar

    best_val_loss    = float("inf")
    best_state       = None
    best_metrics     = None
    best_thresholds  = np.full(N_LABELS, 0.5)
    patience_counter = 0
    PATIENCE         = 1          # [CPU-FIX] 2 → 1: corta antes si no mejora
    history          = {"train_loss": [], "val_loss": [], "val_auc_macro": []}

    epoch_range = tqdm(range(config["num_epochs"]), desc="Épocas", disable=not verbose, unit="ep")

    for epoch in epoch_range:
        if epoch == config["unfreeze_epoch"]:
            model.unfreeze_layers(config["unfreeze_layers"])
            epochs_remaining = config["num_epochs"] - epoch
            optimizer, scheduler = build_optimizer_and_scheduler(
                model, config, len(loader_train),
                num_epochs_override=epochs_remaining
            )

        train_loss = train_one_epoch(
            model, loader_train, optimizer, scheduler, criterion, config,
            epoch_desc=f"Ep{epoch+1} Train"
        )
        val_loss, all_probs, all_labels, all_masks = evaluate(
            model, loader_val, criterion, config, desc=f"Ep{epoch+1} Val"
        )
        val_metrics = compute_multilabel_metrics(all_probs, all_labels, all_masks)

        if scheduler is not None and config["scheduler"] != "onecycle":
            if config["scheduler"] == "plateau":
                scheduler.step(val_loss)
            else:
                scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_auc_macro"].append(val_metrics["macro_AUC"])

        if verbose:
            epoch_range.set_postfix(
                tr_loss=f"{train_loss:.4f}",
                val_loss=f"{val_loss:.4f}",
                AUC=f"{val_metrics['macro_AUC']:.4f}"
            )

        if val_loss < best_val_loss - 1e-4:
            best_val_loss    = val_loss
            best_state       = copy.deepcopy(model.state_dict())
            patience_counter = 0
            if config.get("threshold_search"):
                best_thresholds = find_optimal_thresholds(all_probs, all_labels, all_masks)
            best_metrics = compute_multilabel_metrics(
                all_probs, all_labels, all_masks, thresholds=best_thresholds)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                if verbose:
                    tqdm.write(f"   ⏹ Early stopping en época {epoch+1}.")
                break

    return best_state, best_metrics, best_thresholds, history

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 12: NESTED CROSS-VALIDATION
# [FIX 5] Guardia NaN en AUC del loop interno
# [FIX 6] gc.collect() + cuda.empty_cache() para liberar memoria
# [FIX 8] Progreso explícito con print+flush (tqdm no actualiza en loops anidados
#         bloqueantes — los prints con flush=True garantizan visibilidad inmediata)
# ══════════════════════════════════════════════════════════════════════════════

import gc
import time

K_OUTER = 3
K_INNER = 3

df_cv     = df_train.copy().reset_index(drop=True)
n_samples = len(df_cv)

print("═" * 70)
print("  NESTED CROSS-VALIDATION — ResNet-50 CXR Multilabel")
print("═" * 70)
print(f"  Muestras en CV : {n_samples:,}")
print(f"  K externo      : {K_OUTER}  |  K interno : {K_INNER}")
print(f"  Configs RS     : {len(SAMPLED_CONFIGS)}")
print(f"  Total trains   : {K_OUTER * len(SAMPLED_CONFIGS) * K_INNER} internos "
      f"+ {K_OUTER} finales")
print("═" * 70, flush=True)

outer_results = []
outer_kf      = KFold(n_splits=K_OUTER, shuffle=True, random_state=SEED)

# Contador global para seguimiento
run_total   = K_OUTER * len(SAMPLED_CONFIGS) * K_INNER
run_current = 0
t_global    = time.time()

for outer_fold_idx, (outer_train_idx, outer_test_idx) in enumerate(
    outer_kf.split(np.arange(n_samples))
):
    print(f"\n{'═'*70}", flush=True)
    print(f"  FOLD EXTERNO {outer_fold_idx+1}/{K_OUTER}  "
          f"(train={len(outer_train_idx):,} | test={len(outer_test_idx):,})",
          flush=True)
    print(f"{'═'*70}", flush=True)

    df_outer_train = df_cv.iloc[outer_train_idx].reset_index(drop=True)
    df_outer_test  = df_cv.iloc[outer_test_idx].reset_index(drop=True)

    best_inner_auc = -1.0
    best_config    = SAMPLED_CONFIGS[0]
    inner_kf       = KFold(n_splits=K_INNER, shuffle=True, random_state=SEED)

    for config_idx, config in enumerate(SAMPLED_CONFIGS):

        t_cfg = time.time()
        print(f"\n  ┌─ Config {config_idx+1}/{len(SAMPLED_CONFIGS)} "
              f"[fold ext {outer_fold_idx+1}/{K_OUTER}] "
              f"{'─'*30}", flush=True)
        print(f"  │  lr_head={config['lr_head']}  "
              f"lr_backbone={config['lr_backbone']}  "
              f"dropout={config['dropout_rate']}  "
              f"wd={config['weight_decay']}", flush=True)
        print(f"  │  corr={config['use_label_correlation']}  "
              f"meta={config['use_meta_branch']}  "
              f"policy={config['uncertainty_policy']}", flush=True)

        auc_scores = []

        for inner_fold_idx, (inner_train_idx, inner_val_idx) in enumerate(
            inner_kf.split(np.arange(len(df_outer_train)))
        ):
            run_current += 1
            t_inner = time.time()

            # ── Progreso global ───────────────────────────────────────────────
            elapsed  = time.time() - t_global
            avg_per  = elapsed / run_current if run_current > 1 else 0
            remaining = avg_per * (run_total - run_current)
            eta_min  = int(remaining // 60)
            eta_sec  = int(remaining % 60)

            print(f"  │  [{run_current:3d}/{run_total}] "
                  f"fold interno {inner_fold_idx+1}/{K_INNER}  "
                  f"ETA total: {eta_min}m {eta_sec:02d}s ...",
                  end=" ", flush=True)

            df_inner_train = df_outer_train.iloc[inner_train_idx].reset_index(drop=True)
            df_inner_val   = df_outer_train.iloc[inner_val_idx].reset_index(drop=True)
            
            # En celda 12, después de crear df_inner_train, añade:
            # Subsample del 20% para el loop interno (solo tuning de hiperparámetros)
            df_inner_train = df_inner_train.sample(
                frac=0.20, random_state=SEED
            ).reset_index(drop=True)

            model = CXRResNet50(
                n_labels=N_LABELS,
                dropout_rate=config["dropout_rate"],
                use_label_correlation=config["use_label_correlation"],
                use_meta_branch=config["use_meta_branch"],
                pretrained_source="chexpert",
            ).to(DEVICE)

            _, val_metrics, _, _ = train_model(
                model, df_inner_train, df_inner_val,
                cxr_train_npy, cxr_train_npy,
                config, verbose=False
            )

            # [FIX 5] Guardia NaN
            auc = (val_metrics.get("macro_AUC", float("nan"))
                   if val_metrics is not None else float("nan"))
            auc_scores.append(auc)

            t_inner_min = (time.time() - t_inner) / 60
            print(f"✓  AUC={auc:.4f}  ({t_inner_min:.1f} min)", flush=True)

            # [FIX 6]
            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            gc.collect()

        mean_auc  = float(np.nanmean(auc_scores))
        t_cfg_min = (time.time() - t_cfg) / 60

        print(f"  └─ Config {config_idx+1} completada  "
              f"mean_AUC={mean_auc:.4f}  "
              f"({t_cfg_min:.1f} min total)", flush=True)

        if mean_auc > best_inner_auc:
            best_inner_auc = mean_auc
            best_config    = config
            print(f"     ⭐ Nueva mejor config (AUC={best_inner_auc:.4f})", flush=True)

    # ── Resumen loop interno ───────────────────────────────────────────────────
    print(f"\n  ✅ Mejor config fold externo {outer_fold_idx+1}  "
          f"AUC_inner={best_inner_auc:.4f}", flush=True)
    print(f"     {'─'*50}", flush=True)
    for k, v in best_config.items():
        print(f"     {k:30s} = {v}", flush=True)

    # ── Reentrenamiento final ──────────────────────────────────────────────────
    print(f"\n  🔁 Reentrenando con mejor config sobre train_outer completo "
          f"({len(df_outer_train):,} muestras)...", flush=True)
    t_final = time.time()

    model_final = CXRResNet50(
        n_labels=N_LABELS,
        dropout_rate=best_config["dropout_rate"],
        use_label_correlation=best_config["use_label_correlation"],
        use_meta_branch=best_config["use_meta_branch"],
        pretrained_source="chexpert",
    ).to(DEVICE)

    best_state, _, best_thresholds, history = train_model(
        model_final, df_outer_train, df_outer_test,
        cxr_train_npy, cxr_train_npy,
        best_config, verbose=True
    )

    print(f"  ✓ Reentrenamiento completado en "
          f"{(time.time()-t_final)/60:.1f} min", flush=True)

    # ── Evaluación en test externo ─────────────────────────────────────────────
    model_final.load_state_dict(best_state)
    aug_test  = get_augmentation_pipeline("test")
    ds_test_o = CXRMultilabelDataset(
        df_outer_test, cxr_train_npy, aug_test,
        uncertainty_policy=best_config["uncertainty_policy"]
    )
    loader_test_o = DataLoader(ds_test_o, batch_size=32, shuffle=False, num_workers=0)

    _, test_probs, test_labels_arr, test_masks = evaluate(
        model_final, loader_test_o, criterion, best_config, desc="  Test externo"
    )
    test_metrics = compute_multilabel_metrics(
        test_probs, test_labels_arr, test_masks, thresholds=best_thresholds
    )

    print(f"\n  📊 Test externo fold {outer_fold_idx+1}:", flush=True)
    print(f"     macro_AUC = {test_metrics['macro_AUC']:.4f}  "
          f"macro_F1 = {test_metrics['macro_F1']:.4f}", flush=True)
    print(f"     {'Etiqueta':22s} | {'AUC':>6} | {'F1':>6} | {'N+':>5} | {'N-':>5}",
          flush=True)
    print(f"     {'─'*52}", flush=True)
    for lbl in LABELS:
        m     = test_metrics[lbl]
        auc_s = f"{m['AUC']:.4f}" if not np.isnan(m["AUC"]) else "  N/A"
        print(f"     {lbl:22s} | {auc_s:>6} | {m['F1']:>6.4f} | "
              f"{m['n_pos']:>5} | {m['n_neg']:>5}", flush=True)

    # ── Guardar checkpoint ─────────────────────────────────────────────────────
    ckpt_path = OUTPUT_DIR / f"model_outer_fold{outer_fold_idx+1}.pt"
    torch.save({
        "model_state_dict": best_state,
        "best_config"     : best_config,
        "thresholds"      : best_thresholds.tolist(),
        "test_metrics"    : test_metrics,
        "outer_fold"      : outer_fold_idx + 1,
    }, ckpt_path)
    print(f"  💾 Checkpoint guardado: {ckpt_path}", flush=True)

    outer_results.append({
        "outer_fold"    : outer_fold_idx + 1,
        "best_config"   : best_config,
        "best_inner_auc": best_inner_auc,
        "test_metrics"  : test_metrics,
        "thresholds"    : best_thresholds.tolist(),
        "history"       : history,
        "checkpoint"    : str(ckpt_path),
    })

    del model_final
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()

# ── Resumen final ──────────────────────────────────────────────────────────────
# [FIX 7] recalculadas aquí para robustez de scope entre celdas
auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]
f1_macros  = [r["test_metrics"]["macro_F1"]  for r in outer_results]

t_total_min = (time.time() - t_global) / 60
print("\n" + "═"*70, flush=True)
print("  NESTED CV COMPLETADO", flush=True)
print("═"*70, flush=True)
print(f"  Tiempo total           : {t_total_min:.1f} min", flush=True)
print(f"  AUC macro por fold     : {[f'{a:.4f}' for a in auc_macros]}", flush=True)
print(f"  Media AUC macro        : {np.mean(auc_macros):.4f} ± {np.std(auc_macros):.4f}",
      flush=True)
print(f"  Media F1  macro        : {np.mean(f1_macros):.4f}  ± {np.std(f1_macros):.4f}",
      flush=True)
print("═"*70, flush=True)

══════════════════════════════════════════════════════════════════════
  NESTED CROSS-VALIDATION — ResNet-50 CXR Multilabel
══════════════════════════════════════════════════════════════════════
  Muestras en CV : 10,000
  K externo      : 3  |  K interno : 3
  Configs RS     : 3
  Total trains   : 27 internos + 3 finales
══════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════
  FOLD EXTERNO 1/3  (train=6,666 | test=3,334)
══════════════════════════════════════════════════════════════════════

  ┌─ Config 1/3 [fold ext 1/3] ──────────────────────────────
  │  lr_head=0.0001  lr_backbone=1e-05  dropout=0.4  wd=0.0003
  │  corr=False  meta=True  policy=zeros
  │  [  1/27] fold interno 1/3  ETA total: 0m 00s ...    ✓ Backbone: ResNet-50 pesos CheXpert (torchxrayvision)
   ✓ Backbone: ResNet-50 pesos CheXpert (torchxrayvision)
   ✓ Backbone out dim detectada: 2048  |  needs_GAP: False


Ep1 Train:   0%|          | 0/27 [00:00<?, ?batch/s]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 13: RESUMEN NESTED CV + GUARDADO JSON
# [FIX 7] auc_macros recalculada aquí para robustez de scope entre celdas
# ══════════════════════════════════════════════════════════════════════════════

# [FIX 7] Recalcular por si la celda se ejecuta de forma independiente
auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]
f1_macros  = [r["test_metrics"]["macro_F1"]  for r in outer_results]

print("  AUC por etiqueta (media ± std sobre folds externos):")
for lbl in LABELS:
    aucs = [r["test_metrics"][lbl]["AUC"] for r in outer_results]
    print(f"    {lbl:22s}: {np.nanmean(aucs):.4f} ± {np.nanstd(aucs):.4f}")

summary = {
    "mean_macro_AUC": float(np.mean(auc_macros)),
    "std_macro_AUC" : float(np.std(auc_macros)),
    "mean_macro_F1" : float(np.mean(f1_macros)),
    "std_macro_F1"  : float(np.std(f1_macros)),
    "fold_results"  : [{
        "outer_fold" : r["outer_fold"],
        "macro_AUC"  : r["test_metrics"]["macro_AUC"],
        "macro_F1"   : r["test_metrics"]["macro_F1"],
        "best_config": r["best_config"],
        "thresholds" : r["thresholds"],
        "checkpoint" : r["checkpoint"],
        "per_label"  : {lbl: r["test_metrics"][lbl] for lbl in LABELS},
    } for r in outer_results],
}
p = OUTPUT_DIR / "nested_cv_summary.json"
with open(p, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\n  💾 Resumen guardado: {p}")


  AUC por etiqueta (media ± std sobre folds externos):
    Atelectasis           : nan ± nan
    Cardiomegaly          : nan ± nan
    Edema                 : nan ± nan
    Lung Opacity          : nan ± nan
    No Finding            : nan ± nan
    Pleural Effusion      : nan ± nan

  💾 Resumen guardado: outputs_resnet50_cxr\nested_cv_summary.json


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 14: VISUALIZACIÓN — CURVAS Y AUC POR ETIQUETA
# ══════════════════════════════════════════════════════════════════════════════

# [FIX 7] recalcular por robustez de scope
auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("ResNet-50 CXR — Nested CV Results", fontsize=13, fontweight="bold")

ax = axes[0]
for r in outer_results:
    h = r["history"]
    ax.plot(h["train_loss"], linestyle="--", alpha=0.5, label=f"Fold {r['outer_fold']} train")
    ax.plot(h["val_loss"],                  alpha=0.9, label=f"Fold {r['outer_fold']} val")
ax.set_xlabel("Época"); ax.set_ylabel("Loss"); ax.set_title("Pérdida por Fold")
ax.legend(fontsize=6, ncol=2); ax.grid(alpha=0.3)

ax = axes[1]
for r in outer_results:
    ax.plot(r["history"]["val_auc_macro"], alpha=0.9, label=f"Fold {r['outer_fold']}")
ax.axhline(np.mean(auc_macros), color="red", linestyle=":", label="Media")
ax.set_xlabel("Época"); ax.set_ylabel("AUC macro"); ax.set_title("AUC macro en Val")
ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax = axes[2]
means = [np.nanmean([r["test_metrics"][l]["AUC"] for r in outer_results]) for l in LABELS]
stds  = [np.nanstd( [r["test_metrics"][l]["AUC"] for r in outer_results]) for l in LABELS]
bars  = ax.bar(range(len(LABELS)), means, yerr=stds, color="steelblue", alpha=0.7, capsize=4)
ax.set_xticks(range(len(LABELS)))
ax.set_xticklabels([l[:10] for l in LABELS], rotation=35, ha="right", fontsize=8)
ax.set_ylabel("AUC-ROC"); ax.set_title("AUC por Etiqueta")
ax.set_ylim([0, 1.05])
ax.axhline(np.mean(auc_macros), color="red", linestyle="--", alpha=0.7, label="Macro media")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01,
            f"{val:.3f}", ha="center", fontsize=7)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nested_cv_results.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Figura guardada.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 15: EVALUACIÓN FINAL EN TEST SET OFICIAL
# [FIX 7] auc_macros recalculada al inicio para robustez de scope
# [FIX 2] df_test / cxr_test_npy ya están alineados por hadm_id desde celda 4
# [FIX 2b] Test CSV=464 filas, npy=4640 entradas → join correcto via _npy_idx
# ⚠ EJECUTAR SOLO UNA VEZ — no usar para ajustar nada.
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*70)
print("  EVALUACIÓN FINAL EN TEST SET OFICIAL")
print("  ⚠ Esta evaluación se ejecuta UNA SOLA VEZ.")
print("═"*70)

# [FIX 7]
auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]

best_outer_fold = outer_results[int(np.argmax(auc_macros))]
print(f"\n  Fold seleccionado: {best_outer_fold['outer_fold']}  "
      f"(AUC_outer={auc_macros[int(np.argmax(auc_macros))]:.4f})")

ckpt      = torch.load(best_outer_fold["checkpoint"], map_location=DEVICE)
best_cfg  = ckpt["best_config"]

model_test = CXRResNet50(
    n_labels=N_LABELS,
    dropout_rate=best_cfg["dropout_rate"],
    use_label_correlation=best_cfg["use_label_correlation"],
    use_meta_branch=best_cfg["use_meta_branch"],
    pretrained_source="chexpert",
).to(DEVICE)
model_test.load_state_dict(ckpt["model_state_dict"])

best_thresholds_test = np.array(ckpt["thresholds"])

# [FIX 2] df_test ya tiene _npy_idx correcto, cxr_test_npy es el array completo
aug_test    = get_augmentation_pipeline("test")
ds_test_off = CXRMultilabelDataset(df_test, cxr_test_npy, aug_test,
                                    uncertainty_policy=best_cfg["uncertainty_policy"])
loader_test = DataLoader(ds_test_off, batch_size=32, shuffle=False, num_workers=0)

print(f"  Test samples: {len(ds_test_off):,}  "
      f"[FIX 2b: CSV={len(df_test)} filas alineadas con npy={cxr_test_npy.shape[0]}]")

_, test_probs_f, test_labels_f, test_masks_f = evaluate(
    model_test, loader_test, criterion, best_cfg, desc="Test oficial"
)
test_metrics_final = compute_multilabel_metrics(
    test_probs_f, test_labels_f, test_masks_f, thresholds=best_thresholds_test
)

print("\n  📊 MÉTRICAS FINALES:")
print(f"  {'Etiqueta':22s} | {'AUC':>6} | {'AP':>6} | {'F1':>6} | {'N+':>5} | {'N-':>5}")
print("  " + "─"*60)
for lbl in LABELS:
    m = test_metrics_final[lbl]
    auc_s = f"{m['AUC']:.4f}" if not np.isnan(m["AUC"]) else "  N/A "
    ap_s  = f"{m['AP']:.4f}"  if not np.isnan(m["AP"])  else "  N/A "
    print(f"  {lbl:22s} | {auc_s:>6} | {ap_s:>6} | {m['F1']:>6.4f} | {m['n_pos']:>5} | {m['n_neg']:>5}")
print("  " + "─"*60)
print(f"  {'MACRO':22s} | {test_metrics_final['macro_AUC']:>6.4f} | "
      f"{test_metrics_final['macro_AP']:>6.4f} | {test_metrics_final['macro_F1']:>6.4f}")

with open(OUTPUT_DIR / "final_test_results.json", "w") as f:
    json.dump({
        "macro_AUC" : test_metrics_final["macro_AUC"],
        "macro_AP"  : test_metrics_final["macro_AP"],
        "macro_F1"  : test_metrics_final["macro_F1"],
        "per_label" : {lbl: test_metrics_final[lbl] for lbl in LABELS},
        "best_config": best_cfg,
    }, f, indent=2, default=str)
print(f"\n  💾 Resultados finales guardados.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 16: ANÁLISIS DE EQUIDAD POR SUBGRUPO
# ══════════════════════════════════════════════════════════════════════════════

df_test_r = df_test.reset_index(drop=True)

for col, vals, label in [
    ("gender",         [0, 1],                       "GÉNERO"),
    ("cxr_view",       ["AP", "PA"],                 "VISTA CXR"),
    ("race",           list(RACE_MAP.keys()),        "RAZA"),
    ("admission_type", list(ADMISSION_MAP.keys()),   "TIPO ADMISIÓN"),
]:
    print(f"\n── AUC macro por {label} ──────────────────────────────────────────")
    for v in vals:
        idx = df_test_r[df_test_r[col] == v].index.values
        if len(idx) < 10:
            print(f"  {str(v):25s}: n={len(idx)} (insuficiente)")
            continue
        m = compute_multilabel_metrics(
            test_probs_f[idx], test_labels_f[idx], test_masks_f[idx],
            thresholds=best_thresholds_test
        )
        print(f"  {str(v):25s} (n={len(idx):4d}): "
              f"AUC={m['macro_AUC']:.4f}  F1={m['macro_F1']:.4f}")

print("\n✅ Análisis de equidad completado.")


---
## ✅ Resumen de fixes aplicados en este notebook

| Fix | Celda | Problema | Solución |
|-----|-------|----------|----------|
| **[FIX 1]** | 4, 7 | Imágenes no disponibles como JPGs en disco | Lectura desde `cxr_*.npy` en memoria |
| **[FIX 2]** | 4, 7 | Alineación CSV ↔ npy | Join por `hadm_id` → columna `_npy_idx` |
| **[FIX 2b]**| 4, 15 | Test CSV=464 vs npy=4640 (1/10 del original) | El join por `hadm_id` lo resuelve automáticamente |
| **[FIX 3]** | 10 | `verbose=False` deprecado en PyTorch ≥2.2 | Eliminado de `ReduceLROnPlateau` |
| **[FIX 4]** | 10, 11 | `OneCycleLR` excede pasos al reconstruir tras unfreeze | `num_epochs_override` con épocas restantes |
| **[FIX 5]** | 10, 12 | `macro_AUC` con lista vacía o `None` crasheaba | Guardia `if auc_list` + `nanmean` |
| **[FIX 6]** | 12 | Fragmentación de memoria en runs largos | `gc.collect()` + `cuda.empty_cache()` |
| **[FIX 7]** | 13, 14, 15 | `auc_macros` fuera de scope al re-ejecutar celdas | Recalculada desde `outer_results` al inicio de cada celda |
| **[FIX 8]** | 11 | Sin feedback de progreso durante entrenamiento | `tqdm` en batches, épocas y loop de configs |

## 📌 Próximos pasos
1. **Ablation**: comparar con `use_label_correlation=False` y `use_meta_branch=False`.
2. **ResNet1D para ECG**: misma estructura, señales 1D.
3. **XGBoost para labs**: `labs_percentiles_train.npy` + flags de missingness.
4. **Late fusion stacking**: meta-clasificador sobre las probabilidades de los 3 modelos base.
